# Early Stopping Calibration Experiment

Test different ES parameter settings on a single hyperparameter config to find settings that don't kill training prematurely.

**Problem**: `eps_slope=1e-3` stops all MTS runs at epoch 21-34, before the LR drop at epoch 100.

**Approach**: Train one fixed config with different ES overrides, compare stopping epoch + val NSE.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../'))
os.chdir(os.path.abspath('../../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from UCB_training.UCB_train import UCB_trainer
from UCB_training.UCB_utils import data_dir, get_yaml_path, make_run_stamp, ensure_shared_tree, ensure_absolute_basin_files
from UCB_training.UCB_plotting import extract_losses_from_tensorboard, _simulate_slope_stopping

In [ ]:
# Basin config - start with Hopland, expand later
BASIN = "hopland"
MODE = "mts"
GPU_SETTING = -1
RUN_LABEL = "ES_CALIB"
RUN_STAMP = make_run_stamp()

path_to_csv = data_dir()
path_to_yaml = get_yaml_path(f"{BASIN}_mtslstm2")
_SHARED = ensure_shared_tree(BASIN, MODE)
RUNS_PARENT = _SHARED / "runs" / f"{RUN_LABEL}_{RUN_STAMP}"

# Fixed hyperparams for all runs (no-physics, MTS).
# validate_every=1 ensures every epoch gets a validation loss logged to TB.
# save_weights_every=epochs so only the final checkpoint is kept (ES runs
# also save a checkpoint at the stopping epoch via basetrainer).
BASE_HP = {
    "hidden_size": 128,
    "output_dropout": 0.4,
    "seq_length": {"1D": 90, "1H": 168},
    "num_layers": 1,
    "epochs": 200,
    "batch_size": 64,
    "learning_rate": {0: 0.01, 100: 0.005, 150: 0.001},
    "validate_every": 1,
    "save_weights_every": 200,
}

print(f"Basin: {BASIN}")
print(f"YAML: {path_to_yaml}")
print(f"Runs: {RUNS_PARENT}")

In [ ]:
# ES variants to test
# Each dict overrides the YAML's ES settings via hyperparams
# Note: min_window_gain is set to 0 in slope variants to isolate the effect of eps_slope.
# Without this, min_window_gain=0.01 (YAML default) can independently trigger stopping
# even with very small eps_slope (the check requires BOTH slope AND gain to be improving).
ES_VARIANTS = {
    "no_es": {
        "early_stopping": False,
    },
    "slope_1e-3": {
        "early_stopping": True,
        "early_stopping_mode": "slope",
        "early_stopping_slope_eps_slope": 1e-3,
        "early_stopping_slope_min_epoch": 20,
        "early_stopping_slope_patience": 2,
        "early_stopping_slope_min_window_gain": 0.0,
    },
    "slope_1e-4": {
        "early_stopping": True,
        "early_stopping_mode": "slope",
        "early_stopping_slope_eps_slope": 1e-4,
        "early_stopping_slope_min_epoch": 20,
        "early_stopping_slope_patience": 2,
        "early_stopping_slope_min_window_gain": 0.0,
    },
    "slope_1e-5": {
        "early_stopping": True,
        "early_stopping_mode": "slope",
        "early_stopping_slope_eps_slope": 1e-5,
        "early_stopping_slope_min_epoch": 20,
        "early_stopping_slope_patience": 2,
        "early_stopping_slope_min_window_gain": 0.0,
    },
}

print(f"{len(ES_VARIANTS)} ES variants to test:")
for name in ES_VARIANTS:
    print(f"  - {name}")

In [ ]:
# Run each ES variant sequentially
results = {}

for name, es_overrides in ES_VARIANTS.items():
    print(f"\n{'='*60}")
    print(f"Running: {name}")
    print(f"{'='*60}")

    hp = {**BASE_HP, **es_overrides}
    tag = f"es_calib_{name}"

    trainer = UCB_trainer(
        path_to_csv_folder=path_to_csv, yaml_path=path_to_yaml, hyperparams=hp,
        input_features=None, physics_informed=False, physics_data_file=None,
        hourly=True, extend_train_period=False, gpu=GPU_SETTING, is_mts=True,
        num_ensemble_members=1, adaboost_ensemble=False, bootstrap_model=False,
        verbose=False, runs_parent=str(RUNS_PARENT), run_label=RUN_LABEL,
        run_stamp=RUN_STAMP, experiment_tag=tag)

    ensure_absolute_basin_files(trainer, BASIN)
    trainer.train()
    csv_1d, metrics_1d = trainer.results(period="validation", mts_trk="1D")
    csv_1h, metrics_1h = trainer.results(period="validation", mts_trk="1H")

    # After train(), trainer._model holds the actual NH run dir
    # (e.g. .../testing_run_XXXXXX/) where TB events + model checkpoints live.
    # trainer._runs_parent is the *parent* directory - no TB events there.
    run_dir = Path(trainer._model) if isinstance(trainer._model, (str, Path)) else None

    # Detect actual stopping epoch from the saved model checkpoint.
    # This is more reliable than TB: it accounts for validate_every gaps
    # and the final checkpoint saved by ES on break.
    stop_epoch = trainer._get_last_epoch(run_dir) if run_dir else hp["epochs"]

    # Extract TB loss curves for plotting
    tb = {"train_loss": [], "valid_loss": []}
    if run_dir and run_dir.exists():
        tb = extract_losses_from_tensorboard(run_dir)

    results[name] = {
        "stop_epoch": stop_epoch,
        "NSE_1D": metrics_1d.get("NSE", float("nan")),
        "NSE_1H": metrics_1h.get("NSE", float("nan")),
        "KGE_1D": metrics_1d.get("KGE", float("nan")),
        "KGE_1H": metrics_1h.get("KGE", float("nan")),
        "tb": tb,
        "run_dir": run_dir,
    }
    nse_1d_val = metrics_1d.get("NSE", float("nan"))
    nse_1h_val = metrics_1h.get("NSE", float("nan"))
    print(f"  Stopped: epoch {stop_epoch}")
    print(f"  NSE_1D={nse_1d_val:.4f}  NSE_1H={nse_1h_val:.4f}")

print(f"\nAll {len(results)} runs complete.")

In [ ]:
# Results comparison table
rows = []
for name, r in results.items():
    rows.append({"variant": name, "stop_epoch": r["stop_epoch"],
                 "NSE_1D": round(r["NSE_1D"], 4), "NSE_1H": round(r["NSE_1H"], 4),
                 "KGE_1D": round(r["KGE_1D"], 4), "KGE_1H": round(r["KGE_1H"], 4)})

df_results = pd.DataFrame(rows)
print(f"\n--- {BASIN.upper()} ES Calibration Results ---")
print(df_results.to_string(index=False))

In [ ]:
# Loss curve overlay
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

for (name, r), color in zip(results.items(), colors):
    tb = r["tb"]
    if tb["train_loss"]:
        epochs_t, losses_t = zip(*tb["train_loss"])
        ax1.plot(epochs_t, losses_t, label=f"{name} (ep {r['stop_epoch']})", color=color, alpha=0.8)
    if tb["valid_loss"]:
        epochs_v, losses_v = zip(*tb["valid_loss"])
        ax2.plot(epochs_v, losses_v, label=f"{name} (ep {r['stop_epoch']})", color=color, alpha=0.8)
        ax2.axvline(x=r["stop_epoch"], color=color, linestyle="--", alpha=0.4)

ax1.set_title(f"{BASIN.title()} - Training Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(fontsize=8)
ax1.set_yscale("log")

ax2.set_title(f"{BASIN.title()} - Validation Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend(fontsize=8)
ax2.set_yscale("log")

# Mark LR drops
for ax in (ax1, ax2):
    ax.axvline(x=100, color="gray", linestyle=":", alpha=0.5, label="LR drop 1")
    ax.axvline(x=150, color="gray", linestyle=":", alpha=0.3, label="LR drop 2")

plt.tight_layout()
plt.savefig(f"notebooks/analysis/{BASIN}_es_calibration_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: notebooks/analysis/{BASIN}_es_calibration_loss_curves.png")

In [ ]:
# NSE bar chart comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = list(results.keys())
nse_1d = [results[n]["NSE_1D"] for n in names]
nse_1h = [results[n]["NSE_1H"] for n in names]
stop_eps = [results[n]["stop_epoch"] for n in names]
labels = [f"{n}\n(ep {e})" for n, e in zip(names, stop_eps)]

bars1 = ax1.bar(labels, nse_1d, color=colors[:len(names)])
ax1.set_title(f"{BASIN.title()} - Val NSE 1D")
ax1.set_ylim(0, 1)
for bar, val in zip(bars1, nse_1d):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.3f}", ha="center", fontsize=9)

bars2 = ax2.bar(labels, nse_1h, color=colors[:len(names)])
ax2.set_title(f"{BASIN.title()} - Val NSE 1H")
ax2.set_ylim(0, 1)
for bar, val in zip(bars2, nse_1h):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"notebooks/analysis/{BASIN}_es_calibration_nse.png", dpi=150, bbox_inches="tight")
plt.show()

## Expand to Guerneville
Copy the same experiment for a second basin.

In [ ]:
# Guerneville config
BASIN_2 = "guerneville"
path_to_yaml_2 = get_yaml_path(f"{BASIN_2}_mtslstm2")
_SHARED_2 = ensure_shared_tree(BASIN_2, MODE)
RUNS_PARENT_2 = _SHARED_2 / "runs" / f"{RUN_LABEL}_{RUN_STAMP}"

results_2 = {}

for name, es_overrides in ES_VARIANTS.items():
    print(f"\n{'='*60}")
    print(f"[{BASIN_2}] Running: {name}")
    print(f"{'='*60}")

    hp = {**BASE_HP, **es_overrides}
    tag = f"es_calib_{name}"

    trainer = UCB_trainer(
        path_to_csv_folder=path_to_csv, yaml_path=path_to_yaml_2, hyperparams=hp,
        input_features=None, physics_informed=False, physics_data_file=None,
        hourly=True, extend_train_period=False, gpu=GPU_SETTING, is_mts=True,
        num_ensemble_members=1, adaboost_ensemble=False, bootstrap_model=False,
        verbose=False, runs_parent=str(RUNS_PARENT_2), run_label=RUN_LABEL,
        run_stamp=RUN_STAMP, experiment_tag=tag)

    ensure_absolute_basin_files(trainer, BASIN_2)
    trainer.train()
    csv_1d, metrics_1d = trainer.results(period="validation", mts_trk="1D")
    csv_1h, metrics_1h = trainer.results(period="validation", mts_trk="1H")

    run_dir = Path(trainer._model) if isinstance(trainer._model, (str, Path)) else None
    stop_epoch = trainer._get_last_epoch(run_dir) if run_dir else hp["epochs"]

    tb = {"train_loss": [], "valid_loss": []}
    if run_dir and run_dir.exists():
        tb = extract_losses_from_tensorboard(run_dir)

    results_2[name] = {
        "stop_epoch": stop_epoch,
        "NSE_1D": metrics_1d.get("NSE", float("nan")),
        "NSE_1H": metrics_1h.get("NSE", float("nan")),
        "KGE_1D": metrics_1d.get("KGE", float("nan")),
        "KGE_1H": metrics_1h.get("KGE", float("nan")),
        "tb": tb,
        "run_dir": run_dir,
    }
    nse_1d_val = metrics_1d.get("NSE", float("nan"))
    nse_1h_val = metrics_1h.get("NSE", float("nan"))
    print(f"  Stopped: epoch {stop_epoch}")
    print(f"  NSE_1D={nse_1d_val:.4f}  NSE_1H={nse_1h_val:.4f}")

print(f"\nAll {len(results_2)} Guerneville runs complete.")

In [ ]:
# Guerneville results
rows_2 = []
for name, r in results_2.items():
    rows_2.append({"variant": name, "stop_epoch": r["stop_epoch"],
                   "NSE_1D": round(r["NSE_1D"], 4), "NSE_1H": round(r["NSE_1H"], 4),
                   "KGE_1D": round(r["KGE_1D"], 4), "KGE_1H": round(r["KGE_1H"], 4)})

df_results_2 = pd.DataFrame(rows_2)
print(f"\n--- GUERNEVILLE ES Calibration Results ---")
print(df_results_2.to_string(index=False))

## Patience-based ES
Add patience variants after slope results are in.

In [ ]:
# Patience variants - run on Hopland first
PATIENCE_VARIANTS = {
    "patience_5": {
        "early_stopping": True,
        "early_stopping_mode": "patience",
        "patience_early_stopping": 5,
        "min_delta_early_stopping": 0.0,
        "minimum_epochs_before_early_stopping": 20,
    },
    "patience_10": {
        "early_stopping": True,
        "early_stopping_mode": "patience",
        "patience_early_stopping": 10,
        "min_delta_early_stopping": 0.0,
        "minimum_epochs_before_early_stopping": 20,
    },
}

for name, es_overrides in PATIENCE_VARIANTS.items():
    print(f"\n{'='*60}")
    print(f"[{BASIN}] Running: {name}")
    print(f"{'='*60}")

    hp = {**BASE_HP, **es_overrides}
    tag = f"es_calib_{name}"

    trainer = UCB_trainer(
        path_to_csv_folder=path_to_csv, yaml_path=path_to_yaml, hyperparams=hp,
        input_features=None, physics_informed=False, physics_data_file=None,
        hourly=True, extend_train_period=False, gpu=GPU_SETTING, is_mts=True,
        num_ensemble_members=1, adaboost_ensemble=False, bootstrap_model=False,
        verbose=False, runs_parent=str(RUNS_PARENT), run_label=RUN_LABEL,
        run_stamp=RUN_STAMP, experiment_tag=tag)

    ensure_absolute_basin_files(trainer, BASIN)
    trainer.train()
    csv_1d, metrics_1d = trainer.results(period="validation", mts_trk="1D")
    csv_1h, metrics_1h = trainer.results(period="validation", mts_trk="1H")

    run_dir = Path(trainer._model) if isinstance(trainer._model, (str, Path)) else None
    stop_epoch = trainer._get_last_epoch(run_dir) if run_dir else hp["epochs"]

    tb = {"train_loss": [], "valid_loss": []}
    if run_dir and run_dir.exists():
        tb = extract_losses_from_tensorboard(run_dir)

    results[name] = {
        "stop_epoch": stop_epoch,
        "NSE_1D": metrics_1d.get("NSE", float("nan")),
        "NSE_1H": metrics_1h.get("NSE", float("nan")),
        "KGE_1D": metrics_1d.get("KGE", float("nan")),
        "KGE_1H": metrics_1h.get("KGE", float("nan")),
        "tb": tb,
        "run_dir": run_dir,
    }
    nse_1d_val = metrics_1d.get("NSE", float("nan"))
    nse_1h_val = metrics_1h.get("NSE", float("nan"))
    print(f"  Stopped: epoch {stop_epoch}")
    print(f"  NSE_1D={nse_1d_val:.4f}  NSE_1H={nse_1h_val:.4f}")

# Updated table with all variants
rows_all = []
for name, r in results.items():
    rows_all.append({"variant": name, "stop_epoch": r["stop_epoch"],
                     "NSE_1D": round(r["NSE_1D"], 4), "NSE_1H": round(r["NSE_1H"], 4),
                     "KGE_1D": round(r["KGE_1D"], 4), "KGE_1H": round(r["KGE_1H"], 4)})
df_all = pd.DataFrame(rows_all)
print(f"\n--- ALL {BASIN.upper()} ES Results ---")
print(df_all.to_string(index=False))

In [ ]:
# Guerneville patience variants
for name, es_overrides in PATIENCE_VARIANTS.items():
    print(f"\n{'='*60}")
    print(f"[{BASIN_2}] Running: {name}")
    print(f"{'='*60}")

    hp = {**BASE_HP, **es_overrides}
    tag = f"es_calib_{name}"

    trainer = UCB_trainer(
        path_to_csv_folder=path_to_csv, yaml_path=path_to_yaml_2, hyperparams=hp,
        input_features=None, physics_informed=False, physics_data_file=None,
        hourly=True, extend_train_period=False, gpu=GPU_SETTING, is_mts=True,
        num_ensemble_members=1, adaboost_ensemble=False, bootstrap_model=False,
        verbose=False, runs_parent=str(RUNS_PARENT_2), run_label=RUN_LABEL,
        run_stamp=RUN_STAMP, experiment_tag=tag)

    ensure_absolute_basin_files(trainer, BASIN_2)
    trainer.train()
    csv_1d, metrics_1d = trainer.results(period="validation", mts_trk="1D")
    csv_1h, metrics_1h = trainer.results(period="validation", mts_trk="1H")

    run_dir = Path(trainer._model) if isinstance(trainer._model, (str, Path)) else None
    stop_epoch = trainer._get_last_epoch(run_dir) if run_dir else hp["epochs"]

    tb = {"train_loss": [], "valid_loss": []}
    if run_dir and run_dir.exists():
        tb = extract_losses_from_tensorboard(run_dir)

    results_2[name] = {
        "stop_epoch": stop_epoch,
        "NSE_1D": metrics_1d.get("NSE", float("nan")),
        "NSE_1H": metrics_1h.get("NSE", float("nan")),
        "KGE_1D": metrics_1d.get("KGE", float("nan")),
        "KGE_1H": metrics_1h.get("KGE", float("nan")),
        "tb": tb,
        "run_dir": run_dir,
    }
    nse_1d_val = metrics_1d.get("NSE", float("nan"))
    nse_1h_val = metrics_1h.get("NSE", float("nan"))
    print(f"  Stopped: epoch {stop_epoch}")
    print(f"  NSE_1D={nse_1d_val:.4f}  NSE_1H={nse_1h_val:.4f}")

# All Guerneville results
rows_g_all = []
for name, r in results_2.items():
    rows_g_all.append({"variant": name, "stop_epoch": r["stop_epoch"],
                       "NSE_1D": round(r["NSE_1D"], 4), "NSE_1H": round(r["NSE_1H"], 4),
                       "KGE_1D": round(r["KGE_1D"], 4), "KGE_1H": round(r["KGE_1H"], 4)})
df_g_all = pd.DataFrame(rows_g_all)
print(f"\n--- ALL GUERNEVILLE ES Results ---")
print(df_g_all.to_string(index=False))